In [15]:

import sys
import pandas as pd
from pathlib import Path


import sys
from pathlib import Path

current_dir = Path.cwd()

ROOT_DIR = next(
    (path for path in [current_dir, *current_dir.parents]
     if (path / "src").is_dir() and (path / "utils").is_dir()),
    None
)

if ROOT_DIR is None:
    raise FileNotFoundError("Project root not found.")

if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

print("Project root:", ROOT_DIR)
from src.load_data import load_data
from src.clean_data import clean_missing_values
from src.convert_types import convert_types
from utils.schema import schema

Project root: f:\unsinkable-data


In [16]:
df = load_data(ROOT_DIR / "data" / "raw" / "Titanic_dataset.csv")

print("Dataset shape:",df.shape)
display(df.head())

Dataset shape: (891, 13)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,AgeGroup
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S,Adult
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C,Adult
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S,Adult
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S,Adult
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S,Adult


In [17]:
print(df.columns.tolist())
print("\nMissing values:")
print(df.isnull().sum())

['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked', 'AgeGroup']

Missing values:
PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
AgeGroup         0
dtype: int64


In [18]:
df_clean = clean_missing_values(df, strategy = "median")
df_model = convert_types(df_clean, schema)
print("Raw data:", df.shape)
print("After cleaning:", df_model.shape)
print("\nMissing values:")
print(df_model.isnull().sum())

Raw data: (891, 13)
After cleaning: (889, 12)

Missing values:
PassengerId    0
Survived       0
Pclass         0
Name           0
Sex            0
Age            0
SibSp          0
Parch          0
Ticket         0
Fare           0
Embarked       0
AgeGroup       0
dtype: int64


In [19]:
target = "Survived"
features = [
    "Pclass",
    "Sex",
    "Age",
    "SibSp",
    "Parch",
    "Fare",
    "Embarked"
]

X = df_model[features]
y = df_model[target]

print("X shape:", X.shape)
print("y shape:", y.shape)

display(X.head())
display(y.head())


X shape: (889, 7)
y shape: (889,)


,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,3,male,22.0,1,0,7.2500,S
1,1,female,38.0,1,0,71.2833,C
2,3,female,26.0,0,0,7.9250,S
3,1,female,35.0,1,0,53.1000,S
4,3,male,35.0,0,0,8.0500,S


0    0
1    1
2    1
3    1
4    0
Name: Survived, dtype: int64

In [20]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y
)
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (711, 7)
X_test: (178, 7)
y_train: (711,)
y_test: (178,)


In [21]:
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(
    handle_unknown = "ignore",
    sparse_output=False
)

encoder.fit(X_train[["Sex","Embarked"]])

print(encoder.categories_)

[array(['female', 'male'], dtype=object), array(['C', 'Q', 'S'], dtype=object)]


In [22]:
encoded_train = encoder.transform(
    X_train[["Sex","Embarked"]]
)
print("Encoded shape:", encoded_train.shape)
print(encoded_train[:5])
encoder.get_feature_names_out()

Encoded shape: (711, 5)
[[1. 0. 0. 0. 1.]
 [1. 0. 0. 0. 1.]
 [1. 0. 0. 0. 1.]
 [0. 1. 0. 0. 1.]
 [1. 0. 1. 0. 0.]]


array(['Sex_female', 'Sex_male', 'Embarked_C', 'Embarked_Q', 'Embarked_S'],
      dtype=object)

In [23]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

categorical_features = ["Sex","Embarked"]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        )
    ],
    remainder = "passthrough"
)

X_train_prepared = preprocessor.fit_transform(X_train)
print("Before:", X_train.shape)
print("After:", X_train_prepared.shape)

Before: (711, 7)
After: (711, 10)
